In [1]:
# =========================================================
# BƯỚC 1: CÀI ĐẶT & CHUẨN BỊ DỮ LIỆU GỐC (GOLD SẠCH)
# =========================================================
!pip install -q "huggingface_hub>=0.26.2" polars albumentations pyarrow
!rm -rf /kaggle/working/layout_data/rukopys

In [2]:
import os
import sys
import json
import shutil
import polars as pl
from pathlib import Path
from tqdm import tqdm
from types import ModuleType
import yaml

# --- ĐƯỜNG DẪN TỪ PITO ---
TRAIN_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/train")
TRAIN_META_PATH = TRAIN_DIR / "metadata.jsonl"
MODEL_PATH = "/kaggle/input/models/notpitomon/htd-box-data-preprocessing-doclayoutyolo/pytorch/default/1/Data Preprocessing  DocLayout-Yolo.pt"

OUT_ROOT = Path("/kaggle/working/layout_data/rukopys")
(OUT_ROOT / "images").mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "labels").mkdir(parents=True, exist_ok=True)

# --- CHUYỂN ĐỔI LABEL ---
TYPE2ID = {"handwritten": 0, "printed": 1, "formula": 2, "table": 3, "annotation": 4, "image": 5, "graph": 6}

def convert_coco_to_yolo(x, y, box_w, box_h, img_w, img_h):
    # Dataset gốc dùng định dạng COCO [x, y, w, h]
    cx = (x + box_w / 2.0) / img_w
    cy = (y + box_h / 2.0) / img_h
    nw = box_w / img_w
    nh = box_h / img_h
    return cx, cy, nw, nh

print("🚀 Đang xử lý 1330 ảnh Gold...")
df = pl.read_ndjson(TRAIN_META_PATH)
image_entries = []

for row in tqdm(df.iter_rows(named=True)):
    fname = Path(row["file_name"]).name
    img_w, img_h = row["image_width"], row["image_height"]
    
    in_path = TRAIN_DIR / "images" / fname
    if not in_path.exists(): continue
    
    # Dùng symlink để nhanh và tiết kiệm bộ nhớ
    os.symlink(in_path, OUT_ROOT / "images" / fname)
    
    label_lines = []
    rare_score = 0
    for r in row["regions"] or []:
        t = r["type"]
        if t not in TYPE2ID: continue
        x, y, box_w, box_h = r["bbox"]
        
        # Gọt tọa độ an toàn phòng hờ lỗi tràn viền
        x_safe = max(0.0, float(x))
        y_safe = max(0.0, float(y))
        w_safe = max(1.0, min(float(box_w), img_w - x_safe))
        h_safe = max(1.0, min(float(box_h), img_h - y_safe))
        
        cx, cy, nw, nh = convert_coco_to_yolo(x_safe, y_safe, w_safe, h_safe, img_w, img_h)
        label_lines.append(f"{TYPE2ID[t]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        if t in {"table", "graph", "annotation", "formula"}: rare_score += 1

    with open(OUT_ROOT / "labels" / (Path(fname).stem + ".txt"), "w") as f:
        f.write("\n".join(label_lines))
    
    image_entries.append({"filename": fname, "source": row["source"], "rare_score": rare_score})

🚀 Đang xử lý 1330 ảnh Gold...


1330it [00:05, 240.70it/s]


In [3]:
# =========================================================
# BƯỚC 2: CHIA TRAIN/VAL & OVERSAMPLING (CÂN BẰNG CLASS)
# =========================================================
from sklearn.model_selection import train_test_split
filenames = [e["filename"] for e in image_entries]
sources = [e["source"] for e in image_entries]

train_files, val_files = train_test_split(filenames, test_size=0.15, random_state=42, stratify=sources)
train_set = set(train_files)

oversampled_train = []
for e in image_entries:
    if e["filename"] not in train_set: continue
    repeats = min(1 + e["rare_score"], 5) 
    oversampled_train.extend([e["filename"]] * repeats)

with open(OUT_ROOT / "train.txt", "w") as f:
    for f_n in oversampled_train: f.write(str(OUT_ROOT / "images" / f_n) + "\n")
with open(OUT_ROOT / "val.txt", "w") as f:
    for f_n in val_files: f.write(str(OUT_ROOT / "images" / f_n) + "\n")

In [4]:
# =========================================================
# BƯỚC 3: SETUP REPO & VÁ LỖI (PATCHING)
# =========================================================
%cd /kaggle/working
if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git
%cd DocLayout-YOLO
!pip install -q -e .

# Vá lỗi check_amp
p1 = "doclayout_yolo/utils/checks.py"
with open(p1, "r") as f: c = f.read()
if "def check_amp(model):" in c and "return True" not in c.split("def check_amp(model):")[1][:20]:
    with open(p1, "w") as f: f.write(c.replace("def check_amp(model):", "def check_amp(model):\n    return True\n"))

# Vá lỗi Unpickling (Bảo vệ hàm Strip_optimizer ở cuối epoch)
p2 = "doclayout_yolo/utils/torch_utils.py"
if os.path.exists(p2):
    with open(p2, "r", encoding="utf-8") as f: c = f.read()
    old = 'x = torch.load(f, map_location=torch.device("cpu"))'
    new = 'x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)'
    if old in c:
        with open(p2, "w", encoding="utf-8") as f: f.write(c.replace(old, new))
        print("✅ Đã vá lỗi Unpickling!")

!pip uninstall ray -y -q
sys.modules["doclayout_yolo.utils.callbacks.hub"] = ModuleType("hub"); sys.modules["doclayout_yolo.utils.callbacks.hub"].callbacks = {}

/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 237 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 22.45 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for doclayout_yolo (pyproject.toml) ... done
✅ Đã vá lỗi Unpickling!


In [5]:
# =========================================================
# BƯỚC 4: FINE-TUNING TỪ CHECKPOINT
# =========================================================
yaml_path = "/kaggle/working/rukopys_gold_finetune.yaml"
data_config = {
    "path": str(OUT_ROOT), "train": "train.txt", "val": "val.txt", "nc": 7,
    "names": ["handwritten", "printed", "formula", "table", "annotation", "image", "graph"]
}
with open(yaml_path, "w") as f: yaml.dump(data_config, f, sort_keys=False)

# --- QUICK FIX LỖI .YAML.YAML ĐÂY ---
shutil.copy(yaml_path, yaml_path + ".yaml")

print("\n🔥 BẮT ĐẦU FINE-TUNING ĐỂ NẮN NÓT BOX 🔥")
# Sử dụng lr0 siêu nhỏ và tắt warmup để tinh chỉnh trực tiếp
!WANDB_MODE=disabled RAY_DISABLE_TUNE=1 python train.py \
  --data {yaml_path} \
  --model doclayout_yolo_small \
  --epoch 30 \
  --image-size 1280 \
  --batch-size 8 \
  --project rukopys_gold_finetune \
  --optimizer Adam \
  --lr0 0.0002 \
  --warmup-epochs 0.0 \
  --patience 10 \
  --pretrain "{MODEL_PATH}" \
  --device 0,1 \
  --workers 8


🔥 BẮT ĐẦU FINE-TUNING ĐỂ NẮN NÓT BOX 🔥
New https://pypi.org/project/doclayout_yolo/0.0.4 available 😃 Update with 'pip install -U doclayout_yolo'
Ultralytics YOLOv0.0.2 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                            CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/kaggle/input/models/notpitomon/htd-box-data-preprocessing-doclayoutyolo/pytorch/default/1/Data Preprocessing  DocLayout-Yolo.pt, data=/kaggle/working/rukopys_gold_finetune.yaml.yaml, epochs=30, time=None, patience=10, batch=8, imgsz=1280, save=True, save_period=10, val_period=1, cache=False, device=0,1, workers=8, project=rukopys_gold_finetune, name=rukopys_gold_finetune.yaml_epoch30_imgsz1280_bs8_pretrain_unknown, exist_ok=False, pretrained=True, optimizer=Adam, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=None, amp=True, fraction=1.0, profile=False, freeze=N